# Classical EfficientNet Embedding Extraction

This notebook extracts 1792-dim classical EfficientNet-B4 features for video frames.

### Workflow Stage:
1. **Classical Extraction**: EfficientNet-B4 extracts 1792-dim features.
2. **Output**: Raw embeddings saved for later Quantum-Enhanced training.

In [4]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
from tqdm.auto import tqdm

# Configuration
DATA_DIR = "../data/Videos"
OUTPUT_BASE_DIR = "../embeddings/QEfficient"
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EFFNET_MODEL = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)

print(f"Using device: {DEVICE}")
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

Using device: cuda


## 1. Classical Feature Extraction (EfficientNet-B4)

In [5]:
print("Loading EfficientNet-B4 model...")
model = EFFNET_MODEL.to(DEVICE)
model.classifier = nn.Identity() # Remove the classification head to get features
model.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_effnet_features(image_paths):
    features = []
    for i in range(0, len(image_paths), BATCH_SIZE):
        batch_paths = image_paths[i:i+BATCH_SIZE]
        batch_tensors = []
        for p in batch_paths:
            img = Image.open(p).convert("RGB")
            batch_tensors.append(preprocess(img))
        
        inputs = torch.stack(batch_tensors).to(DEVICE)
        with torch.no_grad():
            batch_features = model(inputs)
        features.append(batch_features.cpu().numpy())
    return np.concatenate(features, axis=0)

Loading EfficientNet-B4 model...


## 2. Main Processing Pipeline

In [6]:
video_ids = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

pbar = tqdm(video_ids, desc="Processing Videos")
for vid in pbar:
    vid_path = os.path.join(DATA_DIR, vid)
    segments = sorted([s for s in os.listdir(vid_path) if os.path.isdir(os.path.join(vid_path, s))])
    
    for seg in segments:
        seg_path = os.path.join(vid_path, seg)
        out_seg_path = os.path.join(OUTPUT_BASE_DIR, vid, seg)
        
        if os.path.exists(os.path.join(out_seg_path, "embeddings.npy")):
            continue
            
        frame_paths = sorted(glob.glob(os.path.join(seg_path, "*.jpg")))
        if not frame_paths: continue
            
        # 1. Classical EfficientNet Extraction (1792-dim)
        effnet_feats = extract_effnet_features(frame_paths)
        
        # Save Raw Classical Results
        os.makedirs(out_seg_path, exist_ok=True)
        np.save(os.path.join(out_seg_path, "embeddings.npy"), effnet_feats)
        
    pbar.set_postfix(video=vid)

print("All Classical EfficientNet embeddings extracted and saved successfully.")

Processing Videos: 100%|██████████| 411/411 [30:36<00:00,  4.47s/it, video=ztgJ33SMons]

All Classical EfficientNet embeddings extracted and saved successfully.
